In [ ]:
%run Setting_Env_Variables_p2.py
google_project_id = %env GOOGLE_CLOUD_PROJECT
bucket = os.getenv("WORKSPACE_BUCKET")

import os
import subprocess
import pandas as pd
from datetime import datetime
import pandas_gbq



In [ ]:
# import hail as hl
# hl.init(gcs_requester_pays_configuration=google_project_id) #, log=f'{bucket}/hail_logs')
# hl.default_reference(new_default_reference = "GRCh38")


In [ ]:
# The below queries represent dataset AllOfUsControlledTierDatasetv9 in cohort LMNA_mutations at 20260721_040756.



# Domain person
person_sql = """

    SELECT
        st.birth_datetime,
        st.ethnicity_concept_id,
        dt0.concept_name AS T_DISP_ethnicity,
        st.gender_concept_id,
        dt1.concept_name AS T_DISP_gender,
        st.person_id,
        st.race_concept_id,
        dt2.concept_name AS T_DISP_race,
        st.self_reported_category_concept_id,
        dt3.concept_name AS T_DISP_self_reported_category,
        st.sex_at_birth_concept_id,
        dt4.concept_name AS T_DISP_sex_at_birth 
    FROM
        `wb-silky-artichoke-2408.C2025Q4R6`.person AS st 
    LEFT JOIN
        `wb-silky-artichoke-2408.C2025Q4R6`.concept AS dt0 
            ON dt0.concept_id = st.ethnicity_concept_id 
    LEFT JOIN
        `wb-silky-artichoke-2408.C2025Q4R6`.concept AS dt1 
            ON dt1.concept_id = st.gender_concept_id 
    LEFT JOIN
        `wb-silky-artichoke-2408.C2025Q4R6`.concept AS dt2 
            ON dt2.concept_id = st.race_concept_id 
    LEFT JOIN
        `wb-silky-artichoke-2408.C2025Q4R6`.concept AS dt3 
            ON dt3.concept_id = st.self_reported_category_concept_id 
    LEFT JOIN
        `wb-silky-artichoke-2408.C2025Q4R6`.concept AS dt4 
            ON dt4.concept_id = st.sex_at_birth_concept_id 
    WHERE
        st.person_id IN (SELECT
            id 
        FROM
            `wb-silky-artichoke-2408.C2025Q4R6_index_061026`.T_ENT_person 
        WHERE
            (id IN (SELECT
                person_id AS primary_id 
            FROM
                `wb-silky-artichoke-2408.C2025Q4R6_index_061026`.T_ENT_conditionOccurrence 
            WHERE
                condition_concept_id IN (SELECT
                    descendant 
                FROM
                    `wb-silky-artichoke-2408.C2025Q4R6_index_061026`.T_HAD_conditionConcept_default 
                WHERE
                    ancestor = 321319 
                UNION
                ALL SELECT
                    321319))) 
                AND (has_whole_genome_variant = true))
"""

# We attempt to use the BigQuery Storage API for high-speed data retrieval.
# If the BQ Storage API is not enabled in the environment or permissions 
# are missing (which may occur if running outside of the Workbench environment), 
# it will fall back to the standard (slower) REST API automatically.
try:
    person_df = pandas_gbq.read_gbq(
        person_sql,
        dialect="standard",
        use_bqstorage_api=True,
        progress_bar_type="tqdm_notebook")
except Exception:
    # Fallback execution if the Storage API is unavailable or unauthorized.
    person_df = pandas_gbq.read_gbq(
        person_sql,
        dialect="standard",
        use_bqstorage_api=False,
        progress_bar_type="tqdm_notebook")

person_df.head(5)
